In [1]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root:", project_root)

Project root: c:\Users\Admin\Desktop\ipo-predictor


In [2]:
import pandas as pd

df = pd.read_csv('../data/raw/ipo_raw.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (395, 11)
Columns: ['IPO Name', 'Listing Date', 'Allotment Price (INR)', 'Listing Return (%)', 'IPO Price', 'Listing Returns (%)', '1 Week Returns (%)', '4 Week Returns (%)', '6 Week Returns (%)', 'IPO Price (INR)', 'IPO Company']


,IPO Name,Listing Date,Allotment Price (INR),Listing Return (%),IPO Price,Listing Returns (%),1 Week Returns (%),4 Week Returns (%),6 Week Returns (%),IPO Price (INR),IPO Company
0,Kissht,5/8/2026,171,22.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Om Power,4/17/2026,175,10.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Powerica,4/2/2026,395,-1.27,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sai Parenteral’s,4/2/2026,392,3.49,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Aeroplane Basmati,4/2/2026,212,-15.09,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# These two columns are the same thing but they are named differently in different year tables
# Combined them into one clean target column

df['listing_gain_pct'] = df['Listing Return (%)'].combine_first(df['Listing Returns (%)'])

print("listing_gain_pct non-null count:", df['listing_gain_pct'].count())
print("Sample values:", df['listing_gain_pct'].dropna().head(10).tolist())

listing_gain_pct non-null count: 395
Sample values: ['22.01', '10.50', '-1.27', '3.49', '-15.09', '-10.43', '11.33', '-27.90', '-7.35', '7.33']


In [4]:
# Same issue: 'IPO Price' and 'Allotment Price (INR)' are the same thing
# Combined into one clean column

df['issue_price'] = df['Allotment Price (INR)'].combine_first(df['IPO Price'])

print("issue_price non-null count:", df['issue_price'].count())
print("Sample values:", df['issue_price'].dropna().head(10).tolist())

issue_price non-null count: 297
Sample values: ['171', '175', '395', '392', '212', '172', '320', '519', '122', '1,352']


In [ ]:
# Dropped all the messy columns, kept only what i cleaned
df_clean = df[[
    'IPO Name',
    'Listing Date',
    'issue_price',
    'listing_gain_pct',
    '1 Week Returns (%)',
    '4 Week Returns (%)',
    '6 Week Returns (%)'
]].copy()

print(f"Shape after column selection: {df_clean.shape}")
df_clean.head()

Shape after column selection: (395, 7)


,IPO Name,Listing Date,issue_price,listing_gain_pct,1 Week Returns (%),4 Week Returns (%),6 Week Returns (%)
0,Kissht,5/8/2026,171,22.01,NaN,NaN,NaN
1,Om Power,4/17/2026,175,10.50,NaN,NaN,NaN
2,Powerica,4/2/2026,395,-1.27,NaN,NaN,NaN
3,Sai Parenteral’s,4/2/2026,392,3.49,NaN,NaN,NaN
4,Aeroplane Basmati,4/2/2026,212,-15.09,NaN,NaN,NaN


In [ ]:
print("Missing before drop:", df_clean.isnull().sum())

# Dropped rows where issue_price or listing_gain_pct is missing
df_clean = df_clean.dropna(subset=['issue_price', 'listing_gain_pct'])

print(f"\nShape after dropping nulls: {df_clean.shape}")
print("Missing after drop:", df_clean.isnull().sum())

Missing before drop: IPO Name               97
Listing Date            0
issue_price            98
listing_gain_pct        0
1 Week Returns (%)    193
4 Week Returns (%)    198
6 Week Returns (%)    200
dtype: int64

Shape after dropping nulls: (297, 7)
Missing after drop: IPO Name               65
Listing Date            0
issue_price             0
listing_gain_pct        0
1 Week Returns (%)    193
4 Week Returns (%)    198
6 Week Returns (%)    200
dtype: int64


In [ ]:
# Converted Listing Date to proper datetime
df_clean['Listing Date'] = pd.to_datetime(df_clean['Listing Date'])

# Converted numeric columns — removed commas first (e.g. "1,258" → 1258)
for col in ['issue_price', 'listing_gain_pct', '1 Week Returns (%)', '4 Week Returns (%)', '6 Week Returns (%)']:
    df_clean[col] = df_clean[col].astype(str).str.replace(',', '').str.strip()
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Renamed columns to clean snake_case
df_clean = df_clean.rename(columns={
    'IPO Name': 'ipo_name',
    'Listing Date': 'listing_date',
    '1 Week Returns (%)': 'return_1w',
    '4 Week Returns (%)': 'return_4w',
    '6 Week Returns (%)': 'return_6w'
})

print(df_clean.dtypes)
print(f"\nShape: {df_clean.shape}")
df_clean.head()

ipo_name                       str
listing_date        datetime64[us]
issue_price                  int64
listing_gain_pct           float64
return_1w                  float64
return_4w                  float64
return_6w                  float64
dtype: object

Shape: (297, 7)


,ipo_name,listing_date,issue_price,listing_gain_pct,return_1w,return_4w,return_6w
0,Kissht,2026-05-08,171,22.01,NaN,NaN,NaN
1,Om Power,2026-04-17,175,10.50,NaN,NaN,NaN
2,Powerica,2026-04-02,395,-1.27,NaN,NaN,NaN
3,Sai Parenteral’s,2026-04-02,392,3.49,NaN,NaN,NaN
4,Aeroplane Basmati,2026-04-02,212,-15.09,NaN,NaN,NaN


In [ ]:
# Extracted useful time features form listing_date
df_clean['listing_year'] = df_clean['listing_date'].dt.year
df_clean['listing_month'] = df_clean['listing_date'].dt.month
df_clean['listing_quarter'] = df_clean['listing_date'].dt.quarter

print(df_clean[['listing_date', 'listing_year', 'listing_month', 'listing_quarter']].head(8))

  listing_date  listing_year  listing_month  listing_quarter
0   2026-05-08          2026              5                2
1   2026-04-17          2026              4                2
2   2026-04-02          2026              4                2
3   2026-04-02          2026              4                2
4   2026-04-02          2026              4                2
5   2026-03-30          2026              3                1
6   2026-03-24          2026              3                1
7   2026-03-23          2026              3                1


In [ ]:
# 1. Regression target: listing_gain_pct (predict exact % gain)
# 2. Classification target: did it give a good listing gain?

def label_gain(pct):
    if pct >= 20:
        return 'strong'    # 20%+ gain
    elif pct >= 5:
        return 'moderate'  # 5-20% gain
    elif pct >= 0:
        return 'weak'      # 0-5% gain
    else:
        return 'loss'      # negative listing

df_clean['gain_category'] = df_clean['listing_gain_pct'].apply(label_gain)

print(df_clean['gain_category'].value_counts())
print(f"\nTotal: {len(df_clean)}")

gain_category
strong      120
loss         82
moderate     59
weak         36
Name: count, dtype: int64

Total: 297


In [10]:
output_path = '../data/processed/ipo_clean.csv'
df_clean.to_csv(output_path, index=False)
print(f"Saved to {output_path}")
print(f"Final shape: {df_clean.shape}")
df_clean.head()

Saved to ../data/processed/ipo_clean.csv
Final shape: (297, 11)


,ipo_name,listing_date,issue_price,listing_gain_pct,return_1w,return_4w,return_6w,listing_year,listing_month,listing_quarter,gain_category
0,Kissht,2026-05-08,171,22.01,NaN,NaN,NaN,2026,5,2,strong
1,Om Power,2026-04-17,175,10.50,NaN,NaN,NaN,2026,4,2,moderate
2,Powerica,2026-04-02,395,-1.27,NaN,NaN,NaN,2026,4,2,loss
3,Sai Parenteral’s,2026-04-02,392,3.49,NaN,NaN,NaN,2026,4,2,weak
4,Aeroplane Basmati,2026-04-02,212,-15.09,NaN,NaN,NaN,2026,4,2,loss
